# Laboratorio 04 — IS-484: Ejercicio Práctico de Laboratorio
## `practicas_programacion.csv`

**Nombre y apellido:** _____________________________

Este notebook es un **archivo independiente** del notebook de la Sección 4
(`Laboratorio04_Clase.ipynb`): no comparten kernel ni variables, así que todo lo
que se necesite de aquel notebook (como la función `normalizar_texto`) debe
copiarse también aquí. Es la base de la nota de laboratorio de la semana: debe
cubrir **todas** las técnicas trabajadas en la Sección 4, no solo algunas de
ellas.

### Parte A: Importación e inspección

In [1]:
import numpy as np
import pandas as pd
import unicodedata
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

# Misma funcion escrita en la Actividad 4.3, Parte A del notebook de la
# Seccion 4: se copia aqui porque este es un notebook independiente.
def normalizar_texto(valor):
    if pd.isna(valor):
        return valor
    texto = valor.strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower()

In [2]:
practicas = pd.read_csv("practicas_programacion.csv")
print("Dimensiones:", practicas.shape)
practicas.dtypes

Dimensiones: (144, 6)


lenguaje                     str
nivel_dificultad             str
uso_ia_asistida              str
num_intentos               int64
tiempo_resolucion_min    float64
puntaje_final            float64
dtype: object

In [3]:
practicas.describe(include="all")

,lenguaje,nivel_dificultad,uso_ia_asistida,num_intentos,tiempo_resolucion_min,puntaje_final
count,144,137,144,144.000000,132.000000,138.000000
unique,4,4,6,NaN,NaN,NaN
top,Python,Medio,No,NaN,NaN,NaN
freq,79,68,78,NaN,NaN,NaN
mean,NaN,NaN,NaN,3.208333,27.409848,14.912319
std,NaN,NaN,NaN,1.572824,26.501251,2.767524
min,NaN,NaN,NaN,0.000000,8.500000,6.700000
25%,NaN,NaN,NaN,2.000000,18.450000,13.025000
50%,NaN,NaN,NaN,3.000000,25.050000,15.300000
75%,NaN,NaN,NaN,4.000000,33.100000,16.850000


In [4]:
practicas.isna().sum()

lenguaje                  0
nivel_dificultad          7
uso_ia_asistida           0
num_intentos              0
tiempo_resolucion_min    12
puntaje_final             6
dtype: int64

**Responder:** ¿qué columnas tienen valores faltantes? ¿qué variables son
numéricas y cuáles categóricas?

*Tu respuesta aquí:*

### Parte B: Limpieza

Normaliza el texto de las columnas categóricas (`lenguaje`, `nivel_dificultad`,
`uso_ia_asistida`) usando la función `normalizar_texto` ya copiada arriba
(la misma escrita en la Actividad 4.3, Parte A del notebook de la Sección 4).
Después, detecta y elimina duplicados
(compara el conteo antes y después de normalizar) y trata los valores atípicos
de `tiempo_resolucion_min` y `num_intentos` con una regla lógica (un tiempo mayor
a 200 minutos, o un número de intentos igual a 0, no son valores creíbles para
este contexto):

In [5]:
columnas_texto_p = ["lenguaje", "nivel_dificultad", "uso_ia_asistida"]
for columna in columnas_texto_p:
    practicas[columna] = practicas[columna].apply(normalizar_texto)

practicas.duplicated().sum()
practicas = practicas.drop_duplicates().reset_index(drop=True)

filtro_tiempo = practicas["tiempo_resolucion_min"] > 200
practicas.loc[filtro_tiempo, "tiempo_resolucion_min"] = np.nan

filtro_intentos = practicas["num_intentos"] == 0
practicas.loc[filtro_intentos, "num_intentos"] = np.nan

Además de la regla lógica, aplica el método del rango intercuartílico (RIC) a
`tiempo_resolucion_min`, para detectar atípicos que una regla fija no alcanza a
cubrir (igual que se hizo con `horas_estudio_semanal` en la Actividad 4.3, Parte C
del notebook de la Sección 4):

In [6]:
# COMPLETAR: calcular Q1, Q3 y el RIC de tiempo_resolucion_min; marcar
# como NaN los valores fuera de [Q1 - 1.5*RIC, Q3 + 1.5*RIC]
# (reproducir el mismo procedimiento de la Actividad 4.3, Parte C)


Finalmente, imputa todos los valores faltantes que queden (los originales, más los generados al tratar los atípicos):

In [7]:
# COMPLETAR: imputar tiempo_resolucion_min, num_intentos y
# puntaje_final con la mediana, y nivel_dificultad con la moda
# (igual que en la Actividad 4.3, Parte D)


### Parte C: Codificación

Codifica `nivel_dificultad` (ordinal: `facil` < `medio` < `dificil`, usando los
valores en minúsculas y sin tilde que dejó la normalización), `uso_ia_asistida`
(binaria) y `lenguaje` (nominal, con One-Hot Encoding). **Sin código de partida:**
reproduce el mismo patrón de la Actividad 4.4, adaptado a las columnas de este
dataset.

No uses `LabelEncoder` directamente sobre `nivel_dificultad`: por ser ordinal, su
orden debe definirse a mano, igual que se hizo con `nivel_socioeconomico`.

### Parte D: Normalización y estandarización

Sobre las columnas numéricas (`num_intentos`, `tiempo_resolucion_min`), aplica y
compara Min-Max y Z-score, igual que en la Actividad 4.5: calcula ambas
transformaciones por separado y responde en qué rango queda cada una.

### Parte E: Separación de datos

Separa $X$ (todas las columnas excepto `puntaje_final` y las categóricas
originales de texto) e $y$ (`puntaje_final`); divide en entrenamiento (80%) y
prueba (20%) con `random_state=42`; y escala las columnas numéricas de
`X_train` y `X_test` de la forma correcta (ajustando el escalador solo con el
entrenamiento), igual que en la Actividad 4.6.